# Spotify Songs' Genre Segmentation
This notebook documents data preparation, exploratory analysis, K-Means clustering, and content-based recommendations. Run `pip install -r requirements.txt` first.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


In [ ]:
df = pd.read_csv('spotify dataset.csv')
features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']
df = df.drop_duplicates(subset='track_id').copy()
df[['track_name','track_artist','track_album_name']] = df[['track_name','track_artist','track_album_name']].fillna('Unknown')
X = StandardScaler().fit_transform(df[features].fillna(df[features].median()))


In [ ]:
scores = {}
for k in range(2, 8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
    scores[k] = silhouette_score(X, labels, sample_size=5000, random_state=42)
best_k = max(scores, key=scores.get)
model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df['cluster'] = model.fit_predict(X)
print('Selected k:', best_k, 'silhouette:', round(scores[best_k], 3))


In [ ]:
pca = PCA(n_components=2, random_state=42)
points = pca.fit_transform(X)
plt.figure(figsize=(10,6))
sns.scatterplot(x=points[:,0], y=points[:,1], hue=df['cluster'], palette='tab10', s=12, alpha=.5)
plt.title('Spotify audio-feature clusters (PCA)')
plt.show()


## Recommendation function
Choose an existing song, then return nearby songs based on standardized audio features within its cluster.


In [ ]:
def recommend(track_name, n=5):
    match = df[df['track_name'].str.lower().eq(track_name.lower())].index
    if len(match) == 0: raise ValueError('Track not found')
    seed = match[0]; cluster = df.loc[seed, 'cluster']
    candidates = df.index[df.cluster.eq(cluster)]
    distance = ((X[candidates] - X[seed])**2).sum(axis=1)**.5
    picks = candidates[np.argsort(distance)[1:n+1]]
    return df.loc[picks, ['track_name','track_artist','playlist_genre','track_popularity','cluster']]
